In [1]:
inference = "random_link_split"
quiried_edge_types = ['indication', 'contraindication', 'off-label use', 'drug_protein', 'drug_effect']
quiried_node_types = ['disease|drug', 'disease|drug', 'disease|drug', 'drug|gene/protein', 'drug|effect/phenotype']

# set edge type flage for test:
edge_flags_val = [1, 2, 3, 4, 5]
edge_flags_test = [1, 2, 3, 4, 5]

Epoch = 1
Repeat = 1
patience_limit = 10

In [2]:
import os
import subprocess
import json
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from torch.optim.lr_scheduler import LambdaLR
import math

from utilities import *
from model import *
from sentence_generation_random_link_split import *

In [3]:
print(os.environ.get('CONDA_DEFAULT_ENV'))
nvidia_smi_output = subprocess.check_output(['nvidia-smi']).decode('utf-8')
print(nvidia_smi_output)
torch.cuda.is_available()

netmedgpt
Mon Oct  6 15:35:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               Off |   00000000:17:00.0 Off |                    0 |
| N/A   36C    P0             81W /  350W |    8120MiB /  81559MiB |      3%      Default |
|                                         |                        |             Disabled |
+-------------------------------------

True

In [4]:
device = torch.device('cuda:0')

with open("/home/bbc8731/NetMedGPT/data/parameters.json", 'r') as file:
    all_param = json.load(file)

data_dir = all_param['files']['data_dir']
best_hyperparam = all_param['best_hyperparam']

os.makedirs(os.path.join(all_param['files']['data_dir'], 'result'), exist_ok = True) 
result_dir = os.path.join(all_param['files']['data_dir'], 'result')

os.makedirs(os.path.join(all_param['files']['data_dir'], "log"), exist_ok=True) # Ensure the folder exists
log_dir = os.path.join(all_param['files']['data_dir'], 'log')

os.makedirs(os.path.join(all_param['files']['data_dir'], "saved_models"), exist_ok=True) # Ensure the folder exists
model_dir = os.path.join(all_param['files']['data_dir'], 'saved_models')

In [5]:
nodes = pd.read_csv(os.path.join(data_dir, 'nodes.csv'), sep= ',')
edge = pd.read_csv(os.path.join(data_dir, "edges.csv")) 
feat = torch.load(os.path.join(data_dir, "embeddings_with_feat.pt"))

In [6]:
# hyperparameters
test_ratio = 0.05
val_ratio = 0.05
seed_base = 42

In [7]:
# dictionary of relations and their node sources
relation_node_types = {
    'protein_protein': ('NCBI', 'NCBI'),
    'drug_protein': ('DrugBank', 'NCBI'),
    'contraindication':('MONDO', 'DrugBank'),
    'indication': ('MONDO', 'DrugBank'),
    'off-label use': ('MONDO', 'DrugBank'),
    'drug_drug': ('DrugBank', 'DrugBank'),
    'phenotype_protein': ('HPO', 'NCBI'),
    'phenotype_phenotype': ('HPO', 'HPO'),
    'disease_phenotype_negative': ('MONDO', 'HPO'),
    'disease_phenotype_positive': ('MONDO', 'HPO'),
    'disease_protein': ('MONDO', 'NCBI'),
    'disease_disease': ('MONDO', 'MONDO'),
    'drug_effect': ('DrugBank', 'HPO'),
    'bioprocess_bioprocess': ('GO', 'GO'),
    'molfunc_molfunc': ('GO', 'GO'),
    'cellcomp_cellcomp': ('GO', 'GO'),
    'molfunc_protein': ('GO', 'NCBI'),
    'cellcomp_protein': ('GO', 'NCBI'),
    'bioprocess_protein': ('GO', 'NCBI'),
    'exposure_protein': ('CTD', 'NCBI'),
    'exposure_disease': ('CTD', 'MONDO'),
    'exposure_exposure': ('CTD', 'CTD'),
    'exposure_bioprocess': ('CTD', 'GO'),
    'exposure_molfunc': ('CTD', 'GO'),
    'exposure_cellcomp': ('CTD', 'GO'),
    'pathway_pathway': ('REACTOME', 'REACTOME'),
    'pathway_protein': ('REACTOME', 'NCBI'),
    'anatomy_anatomy': ('UBERON', 'UBERON'),
    'anatomy_protein_present': ('UBERON', 'NCBI'),
    'anatomy_protein_absent': ('UBERON', 'NCBI'),
}

In [8]:
all_ids = {
    'drug': torch.tensor(
        nodes.loc[nodes['node_type'] == 'drug', 'node_index'].unique(), dtype=torch.int64
    ),
    'gene': torch.tensor(
        nodes.loc[nodes['node_type'] == 'gene/protein', 'node_index'].unique(), dtype=torch.int64
    ),
    'phenotype': torch.tensor(
        nodes.loc[nodes['node_type'] == 'effect/phenotype', 'node_index'].unique(), dtype=torch.int64
    )
}

# Get mapping from node_types to z_index values
node_type_to_z = edge.groupby('node_types')['z_index'].unique().to_dict()

all_ids_node2 = {}

for z in node_type_to_z.get('disease|drug', []):
    all_ids_node2[z] = all_ids['drug']

for z in node_type_to_z.get('drug|gene/protein', []):
    all_ids_node2[z] = all_ids['gene']

for z in node_type_to_z.get('drug|effect/phenotype', []):
    all_ids_node2[z] = all_ids['phenotype']


In [9]:
# other hyperparameters
mask_token = edge['z_index'].max() +1
vocab_size = mask_token + 1

relation_type = list(edge['relation'].unique())
node_types = list(nodes['node_type'].unique())

mask = ['mask']
entity = node_types + relation_type + mask

## find relation_mask_index
relation_index = edge.loc[edge['relation'].isin(relation_type), ['relation', 'z_index']].drop_duplicates()
mask_row = pd.DataFrame([['mask', mask_token]], columns=['relation', 'z_index'])
relation_mask_index = pd.concat([relation_index, mask_row], ignore_index=True)

In [10]:
def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

In [11]:
# Loop over Repeat
LP_result_val_df_all = pd.DataFrame([])
LP_result_test_df_all = pd.DataFrame([])
hit_precision_val_all = pd.DataFrame([])
hit_precision_test_all = pd.DataFrame([])

for i in tqdm(range(Repeat)):
    seed = seed_base + i
    print(f'\nRepeat: {i}, Seed: {seed}')
    patience_counter = 0

    data = sentence_generation(edge, nodes, quiried_node_types, quiried_edge_types, test_ratio, val_ratio, relation_node_types, all_param, seed)

    # walks and validation and test data
    node2vec_walks = data['node2vec_walks']
    val_data_edge_label = data['val_data_edge_label']
    val_data_edge_label_index = data['val_data_edge_label_index_with_relation']
    edge_type_flag_val = data['edge_type_flag_val']
    
    test_data_edge_label = data['test_data_edge_label']
    test_data_edge_label_index = data['test_data_edge_label_index_with_relation']
    edge_type_flag_test = data['edge_type_flag_test']

    log_name = f"biomedformer_atr_{inference}_wl{all_param['node2vec']['walk_length']}_wpn{all_param['node2vec']['walks_per_node']}_dim{best_hyperparam['hidden_channels']}_head{best_hyperparam['nhead']}_lencoder{best_hyperparam['N_encoder_layers']}_bs{best_hyperparam['batch_size']}_lr{best_hyperparam['learning_rate']}_seed{seed}"
    log_file_prob = f"{log_dir}/{log_name}.log"
    
    model_save_path = f"{model_dir}/{log_name}.pt"
    seq_len = (all_param['node2vec']['walk_length']*2)-1   # walk_length_with_relation

    dataset = TensorDataset(node2vec_walks)
    dataloader = DataLoader(dataset, batch_size=best_hyperparam['batch_size'], shuffle=True)

    model = TransformerModel(
        vocab_size,
        best_hyperparam['hidden_channels'],
        best_hyperparam['nhead'],
        best_hyperparam['N_encoder_layers'],
        (all_param['node2vec']['walk_length']*2)-1, #walk_length_with_relation
        device=device,
        feat = feat,
        nodes = nodes,
        entity = entity,
        relation_mask_index = relation_mask_index,
        pos_emb='fixed',
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=best_hyperparam['learning_rate'])
    scaler = GradScaler()

    warmup_steps = int(0.1 * Epoch * len(dataloader))  # 10% warmup
    total_steps = Epoch * len(dataloader)
    
    scheduler = LambdaLR(optimizer, lr_lambda)    
    criterion = nn.CrossEntropyLoss(ignore_index=vocab_size)

    best_auprc = 0.0
    best_model_state = None
    for epoch in range(Epoch):
        model.train()
        epoch_loss = 0

        for batch in dataloader:
            optimizer.zero_grad()
            input_batch = batch[0].to(device)
            masked_input, mask = create_mask(input_batch, vocab_size, mask_token_id=mask_token)
        
            with autocast():  # AMP START
                output = model(masked_input)
                loss = criterion(output[mask], input_batch[mask])
                
            # AMP END
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            batch_loss = loss.item()
            epoch_loss += batch_loss

        avg_loss = epoch_loss / len(dataloader)

        pred_val = sim(val_data_edge_label_index, model, batch_size=best_hyperparam['batch_size'], method='prob', device=device, mask_token=mask_token, seq_len=seq_len)
        pred_val = torch.tensor(pred_val, dtype=torch.float32)
        LP_result_val = link_pred_val(pred_val, val_data_edge_label, edge_type_flag_val, all_param, edge_type = edge_flags_val)
        hit_k_val, precision_k_val = node_level_eval(val_data_edge_label_index, val_data_edge_label, all_ids_node2, model, device=device, mask_token=mask_token, seq_len=seq_len)

        
        auprc = np.array(LP_result_val['auprc']).mean()
        print(auprc)
        
        if (auprc > best_auprc):
            best_auprc = auprc
            best_model_state = model.state_dict()
            best_optimizer_state = optimizer.state_dict()
            best_epoch = epoch + 1
            best_param = best_hyperparam
            loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"patience_counter is {patience_counter}")


        pred_test = sim(test_data_edge_label_index, model, batch_size=best_hyperparam['batch_size'], method='prob', device=device, mask_token=mask_token, seq_len=seq_len)
        pred_test = torch.tensor(pred_test, dtype=torch.float32)
        
        LP_result_test = link_pred_val(pred_test, test_data_edge_label, edge_type_flag_test, all_param, edge_type=edge_flags_test)
        hit_k_test, precision_k_test = node_level_eval(test_data_edge_label_index, test_data_edge_label, all_ids_node2, model, device=device, mask_token=mask_token, seq_len=seq_len)


        # Early stopping
        if (patience_counter >= patience_limit):
            with open(log_file_prob, "a") as f:
                f.write(f"Early stopping triggered at epoch {epoch+1}.\n")
                f.write(f"best_hyperparam: {best_hyperparam}\n")
                f.write(f"best_epoch: {best_epoch}\n")
                break    # break out of epoch loop


    #Save best model
    if best_model_state:
        torch.save(best_model_state, model_save_path)

    # Save final val result to CSV

    columns = ['auc', 'auprc', 'prc@5', 'hits@5', 'prc@100', 'hits@100', 'mrr']
    LP_result_val_df = pd.DataFrame(LP_result_val, columns=columns)
    LP_result_val_df['seed'] = seed
    LP_result_val_df_all = pd.concat([LP_result_val_df_all, LP_result_val_df], ignore_index=True)

    hit_val = pd.DataFrame(hit_k_val, columns=['hit'])
    precision_val = pd.DataFrame(precision_k_val, columns=['precision'])
    hit_precision_val = pd.concat([hit_val, precision_val], axis=1)
    hit_precision_val['seed'] = seed
    hit_precision_val_all = pd.concat([hit_precision_val_all, hit_precision_val], ignore_index=True)

    
    # Save final test result to CSV
    LP_result_test_df = pd.DataFrame(LP_result_test, columns=columns)
    LP_result_test_df['seed'] = seed
    LP_result_test_df_all = pd.concat([LP_result_test_df_all, LP_result_test_df], ignore_index = True)

    hit_test = pd.DataFrame(hit_k_test, columns=['hit'])
    precision_test = pd.DataFrame(precision_k_test, columns=['precision'])
    hit_precision_test = pd.concat([hit_test, precision_test], axis=1)
    hit_precision_test['seed'] = seed
    hit_precision_test_all = pd.concat([hit_precision_test_all, hit_precision_test], ignore_index=True)


log_name_base = f"netmedgpt_atr_{inference}_wl{all_param['node2vec']['walk_length']}_wpn{all_param['node2vec']['walks_per_node']}_dim{best_hyperparam['hidden_channels']}_head{best_hyperparam['nhead']}_lencoder{best_hyperparam['N_encoder_layers']}_bs{best_hyperparam['batch_size']}_lr{best_hyperparam['learning_rate']}"
config_dir = os.path.join(result_dir, log_name_base)
os.makedirs(config_dir, exist_ok=True)

LP_result_val_df_all.to_csv(os.path.join(config_dir, f"LP_val.csv"), index=False)
hit_precision_val_all.to_csv(os.path.join(config_dir, f"hit_precision_val.csv"), index=False)
LP_result_test_df_all.to_csv(os.path.join(config_dir, f"LP_test.csv"), index=False)
hit_precision_test_all.to_csv(os.path.join(config_dir, f"hit_precision_test.csv"), index=False)

summary_val = LP_result_val_df_all.groupby('seed').mean().agg(['mean', 'std'])
summary_test = LP_result_test_df_all.groupby('seed').mean().agg(['mean', 'std'])

summary_val.to_csv(os.path.join(config_dir, f"summary_val.csv"))
summary_test.to_csv(os.path.join(config_dir, f"summary_test.csv"))

with open(os.path.join(config_dir, 'hyperparams.json'), 'w') as f:
    json.dump(best_hyperparam, f, indent=2)


  0%|                                                                                  | 0/1 [00:00<?, ?it/s]


Repeat: 0, Seed: 42
relation: 0
(469, 3)
(470, 3)
relation: 1
(1534, 3)
(1534, 3)
relation: 2
(128, 3)
(129, 3)
relation: 3
(1273, 3)
(1274, 3)
relation: 4
(3239, 3)
(3240, 3)
✅ Transductive isolation successful: no test edge in training.



100%|███████████████████████████████████████████████████████████| 3881250/3881250 [03:02<00:00, 21305.53it/s]
/home/bbc8731/miniconda3/envs/netmedgpt/lib/python3.10/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
  0%|                                                                                  | 0/1 [04:56<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 84.00 MiB. GPU 0 has a total capacity of 79.21 GiB of which 15.62 MiB is free. Process 44190 has 7.92 GiB memory in use. Process 342967 has 61.92 GiB memory in use. Including non-PyTorch memory, this process has 9.34 GiB memory in use. Of the allocated memory 8.13 GiB is allocated by PyTorch, and 450.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# save model checkpoints
checkpoint_path = f"{model_dir}/{log_name}.pt"

# Save the model
checkpoint_dict = {
    'epoch': best_epoch,
    'model_state_dict': best_model_state,
    'optimizer_state_dict': best_optimizer_state,
    'loss': loss,
    'parameters': param
}

torch.save(checkpoint_dict, checkpoint_path)
print(f"Checkpoint saved at {checkpoint_path}")